# Step 1 Target analysis of TA of ideal WL-PSI


### Package imports


In [ ]:
from glotaran.io import load_scheme
from pyglotaran_extras.compat import convert


def _case_study_convert(native_result, scheme):
    """Project native v0.8 results for legacy plotting while retaining native results."""
    import xarray as xr

    compat_result = convert(native_result)
    for dataset_label, dataset in compat_result.data.items():
        optimization_result = native_result.optimization_results[dataset_label]
        global_dimension = optimization_result.meta.global_dimension
        model_dimension = optimization_result.meta.model_dimension
        input_data = optimization_result.input_data
        residual = optimization_result.residuals
        if isinstance(input_data, xr.Dataset):
            input_data = input_data["data"]
        if isinstance(residual, xr.Dataset):
            residual = residual["residual"]
        fitted_data = input_data - residual
        if {"time", "spectral"}.issubset(fitted_data.dims):
            fitted_data = fitted_data.transpose("time", "spectral")
        dataset["fitted_data"] = fitted_data
        data_model = next(
            experiment.datasets[dataset_label]
            for experiment in scheme.experiments.values()
            if dataset_label in experiment.datasets
        )
        weight = xr.ones_like(residual)
        for weight_item in data_model.weights:
            selected = xr.ones_like(residual, dtype=bool)
            if weight_item.global_interval is not None:
                lower, upper = weight_item.global_interval
                selected = selected & (
                    (residual.coords[global_dimension] >= lower)
                    & (residual.coords[global_dimension] <= upper)
                )
            if weight_item.model_interval is not None:
                lower, upper = weight_item.model_interval
                selected = selected & (
                    (residual.coords[model_dimension] >= lower)
                    & (residual.coords[model_dimension] <= upper)
                )
            weight = weight * xr.where(selected, float(weight_item.value), 1.0)
        dataset["weight"] = weight
        dataset["weighted_residual"] = residual * weight
        dataset["clp"] = optimization_result.fit_decomposition.clp.rename(
            amplitude_label="clp_label"
        )
        dataset["matrix"] = optimization_result.fit_decomposition.matrix.rename(
            amplitude_label="clp_label"
        )
        kinetic_elements = [
            element
            for element in optimization_result.elements.values()
            if "compartment" in element.coords
        ]
        if kinetic_elements:
            dataset["species_concentration"] = xr.concat(
                [
                    element["concentrations"].rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            ).transpose(global_dimension, model_dimension, "species")
            dataset["species_associated_spectra"] = xr.concat(
                [element["amplitudes"].rename(compartment="species") for element in kinetic_elements],
                dim="species",
            ).transpose(global_dimension, "species")
            dataset["initial_concentration"] = xr.concat(
                [
                    element["initial_concentrations"]
                    .isel(activation=0, drop=True)
                    .rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
        spectral_elements = [
            element
            for element in optimization_result.elements.values()
            if "shape" in element.coords
        ]
        if spectral_elements:
            dataset["species_spectra"] = xr.concat(
                [
                    element["concentrations"].squeeze(drop=True).rename(shape="species")
                    for element in spectral_elements
                ],
                dim="species",
            ).transpose(model_dimension, "species")
        dataset.attrs["dataset_scale"] = optimization_result.meta.scale
    return compat_result


def _case_study_matrix_markdown(scheme, element_label, compartments=None):
    """Render a symbolic v0.8 kinetic rate map for legacy notebook display cells."""
    import pandas as pd

    element = scheme.library[element_label]
    compartments = list(compartments or element.compartments)
    table = [["" for _ in compartments] for _ in compartments]
    for (to_compartment, from_compartment), rate in element.rates.items():
        if to_compartment in compartments and from_compartment in compartments:
            table[compartments.index(to_compartment)][compartments.index(from_compartment)] = str(rate)
    return pd.DataFrame(table, index=compartments, columns=compartments).to_markdown()


from glotaran.io import load_parameters, save_result
from pyglotaran_extras import plot_overview, plot_data_overview
from pyglotaran_extras import plot_fitted_traces, select_plot_wavelengths
from pyglotaran_extras.inspect import show_a_matrixes
from pyglotaran_extras.plotting.style import PlotStyle
from pyglotaran_extras.plotting.style import ColorCode
from cycler import cycler

### Data inspection


In [ ]:
DATA_PATH3 = 'data/synWTred_idealc.ascii'
DATA_PATH4 = 'data/synWTred_ideald.ascii'

In [ ]:
(fig, axes) = plot_data_overview(DATA_PATH3, nr_of_data_svd_vectors=5, linlog=False, cmap='seismic', vmin=-7, vmax=7, use_svd_number=True)

In [ ]:
(fig, axes) = plot_data_overview(DATA_PATH4, nr_of_data_svd_vectors=5, linlog=False, linthresh=10, cmap='seismic', vmin=-7, vmax=7, use_svd_number=True)

## Target Analysis


### Model specification


#### Model file


In [ ]:
target_model_path = 'models/ideal_target_model_PSI_TA_v08.yml'

#### Parameters file


In [ ]:
target_parameters_path = 'models/ideal_target_parameters_PSI_TA.csv'
start_parameters = load_parameters(target_parameters_path)

### Define the analysis scheme and optimize

A scheme is a collection of a model, parameters and data, along with options for optimization.


In [ ]:
target_scheme = load_scheme(target_model_path)
target_scheme_parameters = start_parameters
target_scheme_datasets = {'700TR1': DATA_PATH3, '700TR2': DATA_PATH4}
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=7, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=7, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result = _case_study_convert(target_result_native, target_scheme)

<sub>For reference, after 6 iterations the final cost is expected to be 7.7517e-05.</sub>

To save the results of the optimization we can use the `save_result` command.

Because it saves _everything_ it consumes about 15MB of disk space per save.


In [ ]:
save_result(result=target_result_native, result_path='results/ideal/result.yaml', allow_overwrite=True)

### Results and parameters


In [ ]:
target_result

In [ ]:
target_result.optimized_parameters

## Result plots


<sub>Note: The color scheme of the plots in this notebook may not match published figures.</sub>


## Fit quality


In [ ]:
target_result_TA = (target_result.data['700TR1'], target_result.data['700TR2'])
wavelengths = select_plot_wavelengths(target_result_TA, equidistant_wavelengths=True)
fig_traces = plot_fitted_traces(target_result_TA, wavelengths, linlog=True, linthresh=1)

The above command `plot_fitted_traces` is used to plot a selection of traces for a set of wavelengths (autogenerated using the `select_plot_wavelengths` function).


## Overview 700 exc


Note that the final rms error is 3.5E-5, which is almost a million times smaller than the largest signal, a signal to noise ratio that is of course experimentally impossible. This residual matrix still shows some structure in the overview plots below.


In [ ]:
fig_700tr1 = plot_overview(target_result.data['700TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

In [ ]:
fig_700tr2 = plot_overview(target_result.data['700TR2'], nr_of_data_svd_vectors=5, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']), use_svd_number=True, das_cycler=PlotStyle().cycler, svd_cycler=PlotStyle().cycler)

## Plot result for interpretation


In [ ]:
from custom_plotting import plot_concentration_and_spectra
custom_cycler = cycler(color=['g', 'r', 'k', ColorCode.cyan, 'b'])
custom_cycler2 = cycler(color=['tab:grey', 'tab:orange', ColorCode.green, ColorCode.turquoise, 'm'])
(fig, axes) = plot_concentration_and_spectra([target_result.data['700TR1'], target_result.data['700TR2']], cycler=custom_cycler, das_cycler=custom_cycler2)

### Amplitude matrices


In [ ]:
show_a_matrixes(target_result)

In [ ]:
_case_study_matrix_markdown(target_scheme, 'complex700', ['Bulk', 'Red', 'RC', 'RP1', 'RP2'])

In [ ]:
_case_study_matrix_markdown(target_scheme, 'complex700', ['Bulk', 'Red', 'RC', 'RP1', 'RP2'])